In [ ]:
#mount drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install segmentation-models-pytorch albumentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.9 MB/s eta 0:00:00


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
import cv2
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.models.segmentation import fcn_resnet50
import torch.nn as nn
import segmentation_models_pytorch as smp
import numpy as np

In [ ]:
train_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: Combined/images")
train_mask_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/train/Second Batch: Combined/masks")
val_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/val/val_images")
val_mask_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/val/val_masks")
test_image_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/test/test_images")
test_masks_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/test/test_masks")
export_dir = Path("/content/drive/MyDrive/Masters/SP2026/Classes/E6692.2025Spring.ETFJ.et2833/run2_split_dataset/test")

In [ ]:
def count_images(folder):
    image_extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    return sum(1 for f in folder.iterdir() if f.suffix.lower() in image_extensions)

# Print counts
print("Train images:", count_images(train_image_dir))
print("Train masks:", count_images(train_mask_dir))

print("Validation images:", count_images(val_image_dir))
print("Validation masks:", count_images(val_mask_dir))

print("Test images:", count_images(test_image_dir))
print("Test masks:", count_images(test_masks_dir))

Train images: 350
Train masks: 348
Validation images: 168
Validation masks: 216
Test images: 168
Test masks: 165


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from utils.liche_dataset import LichenDataset


In [ ]:
#Datasets and Loaders

IMAGE_SIZE = (512, 512)
BATCH_SIZE = 4

train_dataset = LichenDataset(train_image_dir, train_mask_dir)
img, mask = train_dataset[0]

val_dataset = LichenDataset(val_image_dir, val_mask_dir)
img, mask = val_dataset[0]

test_dataset = LichenDataset(test_image_dir, test_masks_dir)
img, mask = test_dataset[0]



train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

⚠️ Missing mask for: lichen_133244390.jpeg
⚠️ Missing mask for: lichen_33879765.jpg
✅ Matched image-mask pairs: 348
⚠️ Missing mask for: lichen_192189784.jpg
⚠️ Missing mask for: lichen_325068990.jpg
✅ Matched image-mask pairs: 166
⚠️ Missing mask for: lichen_134788355.jpeg
⚠️ Missing mask for: lichen_141700394.jpeg
⚠️ Missing mask for: lichen_297129001.jpg
✅ Matched image-mask pairs: 165
348 166 165


In [ ]:
#U_Net

#encoder = resnet34

unet_resnet34 = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_unet_resnet34 = torch.optim.Adam(
    unet_resnet34.parameters(),
    lr=1e-4
)

UNET_RESNET34_SAVE_PATH = (
    f"{export_dir}/2unet_resnet34_lichen.pth"
)

#encoder = resnet18

unet_resnet18 = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_unet_resnet18 = torch.optim.Adam(
    unet_resnet18.parameters(),
    lr=1e-4
)

UNET_RESNET18_SAVE_PATH = (
    f"{export_dir}/2unet_resnet18_lichen.pth"
)

#encoder = mobilnet

unet_mobilenet = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_unet_mobilenet = torch.optim.Adam(
    unet_mobilenet.parameters(),
    lr=1e-4
)

UNET_MOBILENET_SAVE_PATH = (
    f"{export_dir}/2unet_mobilenetv2_lichen.pth"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

In [ ]:
#Deeplab


#encoder = resnet34
deeplab_resnet34 = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_deeplab_resnet34 = torch.optim.Adam(
    deeplab_resnet34.parameters(),
    lr=1e-4
)

DEEPLAB_RESNET34_SAVE_PATH = (
    f"{export_dir}/2deeplab_resnet34_lichen.pth"
)

#encoder = resnet18
deeplab_resnet18 = smp.DeepLabV3Plus(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_deeplab_resnet18 = torch.optim.Adam(
    deeplab_resnet18.parameters(),
    lr=1e-4
)

DEEPLAB_RESNET18_SAVE_PATH = (
    f"{export_dir}/2deeplab_resnet18_lichen.pth"
)
#encoder = mobilnet
deeplab_mobilenet = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

optimizer_deeplab_mobilenet = torch.optim.Adam(
    deeplab_mobilenet.parameters(),
    lr=1e-4
)

DEEPLAB_MOBILENET_SAVE_PATH = (
    f"{export_dir}/2deeplab_mobilenetv2_lichen.pth"
)


In [ ]:
from utils_training import *

In [ ]:
EPOCHS = 30


train_model(
    unet_resnet34,
    train_loader,
    val_loader,
    optimizer_unet_resnet34,
    UNET_RESNET34_SAVE_PATH,
    EPOCHS
)

print("done with unet_resnet34")
train_model(
    unet_resnet18,
    train_loader,
    val_loader,
    optimizer_unet_resnet18,
    UNET_RESNET18_SAVE_PATH,
    EPOCHS
)
print("done with unet_resnet18")
train_model(
    unet_mobilenet,
    train_loader,
    val_loader,
    optimizer_unet_mobilenet,
    UNET_MOBILENET_SAVE_PATH,
    EPOCHS
)
print("done with unet_mobilenet")


train_model(
    deeplab_resnet34,
    train_loader,
    val_loader,
    optimizer_deeplab_resnet34,
    DEEPLAB_RESNET34_SAVE_PATH,
    EPOCHS
)
print("done with deeplab_resnet34")
train_model(
    deeplab_resnet18,
    train_loader,
    val_loader,
    optimizer_deeplab_resnet18,
    DEEPLAB_RESNET18_SAVE_PATH,
    EPOCHS
)
print("done with deeplab_resnet18")
train_model(
    deeplab_mobilenet,
    train_loader,
    val_loader,
    optimizer_deeplab_mobilenet,
    DEEPLAB_MOBILENET_SAVE_PATH,
    EPOCHS
)

print("done with deeplab_mobilenet")

Epoch 1/30
Train Loss: 0.9696
Val Loss:   0.9285
Saved best model!

Epoch 2/30
Train Loss: 0.6917
Val Loss:   0.7562
Saved best model!

Epoch 3/30
Train Loss: 0.6250
Val Loss:   0.7293
Saved best model!

Epoch 4/30
Train Loss: 0.4882
Val Loss:   0.7315

Epoch 5/30
Train Loss: 0.4447
Val Loss:   0.7142
Saved best model!

Epoch 6/30
Train Loss: 0.3925
Val Loss:   0.8360

Epoch 7/30
Train Loss: 0.3812
Val Loss:   0.7625

Epoch 8/30
Train Loss: 0.3424
Val Loss:   0.7454

Epoch 9/30
Train Loss: 0.2891
Val Loss:   0.7567

Epoch 10/30
Train Loss: 0.2673
Val Loss:   0.7297

Epoch 11/30
Train Loss: 0.2296
Val Loss:   0.7220

Epoch 12/30
Train Loss: 0.2146
Val Loss:   0.7494

Epoch 13/30
Train Loss: 0.1995
Val Loss:   0.8101

Epoch 14/30
Train Loss: 0.2001
Val Loss:   0.7900

Epoch 15/30
Train Loss: 0.1852
Val Loss:   0.8099

Epoch 16/30
Train Loss: 0.1697
Val Loss:   0.8271

Epoch 17/30
Train Loss: 0.1597
Val Loss:   0.7860

Epoch 18/30
Train Loss: 0.1534
Val Loss:   0.8287

Epoch 19/30
Train L

In [ ]:
#metrics for comparison

from utils.evaluate_segmentation import *



In [ ]:
#load best

unet_resnet34.load_state_dict(
    torch.load(UNET_RESNET34_SAVE_PATH, map_location=device)
)

unet_resnet18.load_state_dict(
    torch.load(UNET_RESNET18_SAVE_PATH, map_location=device)
)

unet_mobilenet.load_state_dict(
    torch.load(UNET_MOBILENET_SAVE_PATH, map_location=device)
)

deeplab_resnet34.load_state_dict(
    torch.load(DEEPLAB_RESNET34_SAVE_PATH, map_location=device)
)

deeplab_resnet18.load_state_dict(
    torch.load(DEEPLAB_RESNET18_SAVE_PATH, map_location=device)
)

deeplab_mobilenet.load_state_dict(
    torch.load(DEEPLAB_MOBILENET_SAVE_PATH, map_location=device)
)




val_results = {
    "U-Net ResNet34": evaluate_segmentation(
        unet_resnet34,
        val_loader
    ),

    "U-Net ResNet18": evaluate_segmentation(
        unet_resnet18,
        val_loader
    ),

    "U-Net MobileNetV2": evaluate_segmentation(
        unet_mobilenet,
        val_loader
    ),

    "DeepLabV3+ ResNet34": evaluate_segmentation(
        deeplab_resnet34,
        val_loader
    ),

    "DeepLabV3+ ResNet18": evaluate_segmentation(
        deeplab_resnet18,
        val_loader
    ),

    "DeepLabV3+ MobileNetV2": evaluate_segmentation(
        deeplab_mobilenet,
        val_loader
    ),
}



test_results = {
    "U-Net ResNet34": evaluate_segmentation(
        unet_resnet34,
        test_loader
    ),

    "U-Net ResNet18": evaluate_segmentation(
        unet_resnet18,
        test_loader
    ),

    "U-Net MobileNetV2": evaluate_segmentation(
        unet_mobilenet,
        test_loader
    ),

    "DeepLabV3+ ResNet34": evaluate_segmentation(
        deeplab_resnet34,
        test_loader
    ),

    "DeepLabV3+ ResNet18": evaluate_segmentation(
        deeplab_resnet18,
        test_loader
    ),

    "DeepLabV3+ MobileNetV2": evaluate_segmentation(
        deeplab_mobilenet,
        test_loader
    ),
}


#display

import pandas as pd

val_df = pd.DataFrame(val_results).T
test_df = pd.DataFrame(test_results).T

print("Validation Results")
display(val_df)

print("Test Results")
display(test_df)

Validation Results


,IoU,Dice,Precision,Recall,Accuracy
U-Net ResNet34,0.625859,0.769881,0.718658,0.828966,0.832876
U-Net ResNet18,0.625147,0.769342,0.778972,0.759948,0.846324
U-Net MobileNetV2,0.629499,0.772629,0.771173,0.774090,0.846351
DeepLabV3+ ResNet34,0.626766,0.770567,0.806435,0.737753,0.851840
DeepLabV3+ ResNet18,0.610302,0.757997,0.782989,0.734550,0.841821
DeepLabV3+ MobileNetV2,0.635187,0.776898,0.754611,0.800542,0.844941


Test Results


,IoU,Dice,Precision,Recall,Accuracy
U-Net ResNet34,0.622979,0.767698,0.692984,0.860470,0.830383
U-Net ResNet18,0.604969,0.753870,0.737335,0.771163,0.835984
U-Net MobileNetV2,0.607268,0.755652,0.747751,0.763722,0.839123
DeepLabV3+ ResNet34,0.626938,0.770697,0.771761,0.769635,0.850829
DeepLabV3+ ResNet18,0.588784,0.741176,0.744795,0.737591,0.832208
DeepLabV3+ MobileNetV2,0.625890,0.769904,0.743194,0.798606,0.844520


In [ ]:
def predict_mask(model, img, threshold=0.5):
    model.eval()

    with torch.no_grad():
        img_batch = img.unsqueeze(0).to(device)

        logits = get_logits(model, img_batch)
        probs = torch.sigmoid(logits)

        pred_mask = (probs > threshold).float()

    return pred_mask.cpu().squeeze()

In [ ]:
def find_lichen_quadrant(mask):


    mask = mask.squeeze()  # force to [H, W]

    H, W = mask.shape

    mid_H = H // 2
    mid_W = W // 2

    quadrants = {
        "top-left": mask[:mid_H, :mid_W],
        "top-right": mask[:mid_H, mid_W:],
        "bottom-left": mask[mid_H:, :mid_W],
        "bottom-right": mask[mid_H:, mid_W:]
    }

    counts = {
        name: quadrant.sum().item()
        for name, quadrant in quadrants.items()
    }

    best_quadrant = max(counts, key=counts.get)

    boxes = {
        "top-left": (0, 0, mid_W, mid_H),
        "top-right": (mid_W, 0, W - mid_W, mid_H),
        "bottom-left": (0, mid_H, mid_W, H - mid_H),
        "bottom-right": (mid_W, mid_H, W - mid_W, H - mid_H)
    }

    return best_quadrant, boxes[best_quadrant], counts

In [ ]:
def extract_quadrant(tensor, quadrant):

    # If tensor is [1, H, W] or [3, H, W]
    if tensor.ndim == 3:
        H = tensor.shape[1]
        W = tensor.shape[2]

    # If tensor is [H, W]
    elif tensor.ndim == 2:
        H = tensor.shape[0]
        W = tensor.shape[1]

    else:
        raise ValueError(f"Unexpected tensor shape: {tensor.shape}")

    mid_H = H // 2
    mid_W = W // 2

    if quadrant == "top-left":
        return tensor[..., :mid_H, :mid_W]

    elif quadrant == "top-right":
        return tensor[..., :mid_H, mid_W:]

    elif quadrant == "bottom-left":
        return tensor[..., mid_H:, :mid_W]

    elif quadrant == "bottom-right":
        return tensor[..., mid_H:, mid_W:]

    else:
        raise ValueError(f"Unknown quadrant: {quadrant}")

In [ ]:
def process_predicted_masks(model, dataset, save_dir, model_name, threshold=0.5):

    save_dir = Path(save_dir) / model_name
    save_dir.mkdir(parents=True, exist_ok=True)

    model.eval()

    for idx in range(len(dataset)):

        img, true_mask = dataset[idx]

        # 1. Predict binary lichen mask
        pred_mask = predict_mask(
            model,
            img,
            threshold=threshold
        )

        # 2. Find quadrant with most predicted lichen
        best_quadrant, box, counts = find_lichen_quadrant(
            pred_mask
        )

        # 3. Extract that quadrant from predicted mask
        pred_quadrant_tensor = extract_quadrant(
            pred_mask,
            best_quadrant
        )

        # 4. Optionally also extract the image quadrant
        img_quadrant_tensor = extract_quadrant_image(
            img,
            best_quadrant
        )

        # 5. Save everything useful
        save_dict = {
            "predicted_mask_quadrant": pred_quadrant_tensor,
            "image_quadrant": img_quadrant_tensor,
            "best_quadrant": best_quadrant,
            "box": box,
            "counts": counts,
            "threshold": threshold
        }

        save_path = save_dir / f"sample_{idx}_{best_quadrant}.pt"

        torch.save(save_dict, save_path)

    print(f"Saved predicted quadrant tensors to: {save_dir}")

In [ ]:
def extract_quadrant_image(img, quadrant):
    """
    img shape: [3, H, W]
    """

    C, H, W = img.shape

    mid_H = H // 2
    mid_W = W // 2

    if quadrant == "top-left":
        return img[:, :mid_H, :mid_W]

    elif quadrant == "top-right":
        return img[:, :mid_H, mid_W:]

    elif quadrant == "bottom-left":
        return img[:, mid_H:, :mid_W]

    elif quadrant == "bottom-right":
        return img[:, mid_H:, mid_W:]

In [ ]:
PRED_SAVE_DIR = "/content/drive/MyDrive/Masters/SP2026/Classes"

models = {
    "unet_resnet34": unet_resnet34,
    "unet_resnet18": unet_resnet18,
    "unet_mobilenetv2": unet_mobilenet,
    "deeplab_resnet34": deeplab_resnet34,
    "deeplab_resnet18": deeplab_resnet18,
    "deeplab_mobilenetv2": deeplab_mobilenet,
}

for model_name, model in models.items():

    process_predicted_masks(
        model=model,
        dataset=test_dataset,
        save_dir=PRED_SAVE_DIR,
        model_name=model_name,
        threshold=0.5
    )

Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/unet_resnet34
Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/unet_resnet18
Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/unet_mobilenetv2
Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/deeplab_resnet34
Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/deeplab_resnet18
Saved predicted quadrant tensors to: /content/drive/MyDrive/Masters/SP2026/Classes/deeplab_mobilenetv2
